# 🧩 Mini-Lab: Metadata Filtering

**Module 5: Embeddings & Document Processing** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** what metadata is and why it enhances document retrieval
2. **Explain** how to extract and store metadata alongside document embeddings
3. **Compare** filtered vs. unfiltered search results using metadata criteria

## Target Concepts

| Concept | Description |
|---------|-------------|
| Semantic Chunking | Splitting text based on meaning boundaries rather than fixed sizes |
| Metadata Extraction | Identifying and storing structured attributes (category, date, author) with documents |

In [1]:
# Install chromadb if needed
# !pip install chromadb openai

from openai import OpenAI
import chromadb
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

client = OpenAI()

## 1. What is Metadata?

Metadata is additional information about a document (author, date, category, etc.).

In [2]:
# Documents with metadata
documents = [
    {
        "text": "Python 3.11 introduces significant performance improvements.",
        "metadata": {"category": "programming", "language": "python", "year": 2023}
    },
    {
        "text": "AI-powered code completion tools are transforming modern software development.",
        "metadata": {"category": "AI", "language": "english", "year": 2023}
    },
    {
        "text": "React 18 adds concurrent rendering features.",
        "metadata": {"category": "programming", "language": "javascript", "year": 2022}
    },
    {
        "text": "Large language models can now generate and refactor code with new programming paradigms.",
        "metadata": {"category": "AI", "language": "english", "year": 2024}
    },
    {
        "text": "TypeScript adds static typing to JavaScript.",
        "metadata": {"category": "programming", "language": "typescript", "year": 2012}
    },
    {
        "text": "Neural networks enable intelligent auto-complete and modern coding features in IDEs.",
        "metadata": {"category": "AI", "language": "english", "year": 2023}
    }
]

md("### Sample Documents with Metadata:")
for i, doc in enumerate(documents, 1):
    md(f"\n**Document {i}:**")
    md(f"- Text: {doc['text']}")
    md(f"- Metadata: `{doc['metadata']}`")

### Sample Documents with Metadata:


**Document 1:**

- Text: Python 3.11 introduces significant performance improvements.

- Metadata: `{'category': 'programming', 'language': 'python', 'year': 2023}`


**Document 2:**

...

- Text: TypeScript adds static typing to JavaScript.

- Metadata: `{'category': 'programming', 'language': 'typescript', 'year': 2012}`


**Document 6:**

- Text: Neural networks enable intelligent auto-complete and modern coding features in IDEs.

- Metadata: `{'category': 'AI', 'language': 'english', 'year': 2023}`

## 2. Storing Documents with Metadata

In [3]:
# Initialize vector database
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="docs_with_metadata")

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# Add documents with metadata
for i, doc in enumerate(documents):
    embedding = get_embedding(doc['text'])
    collection.add(
        embeddings=[embedding],
        documents=[doc['text']],
        metadatas=[doc['metadata']],
        ids=[f"doc_{i}"]
    )

md(f"✅ Added **{len(documents)} documents** with metadata to vector database")

✅ Added **6 documents** with metadata to vector database

## 3. Search with Metadata Filtering

Filter results to only specific categories or attributes.

In [5]:
query = "modern programming features"
query_embedding = get_embedding(query)

# Search without filtering
results_all = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

md(f"**Query:** {query}")
md("\n### Results (No Filtering):")
for i, (doc, meta) in enumerate(zip(results_all['documents'][0], results_all['metadatas'][0]), 1):
    md(f"\n**{i}.** {doc}")
    md(f"   - Category: {meta['category']}, Language: {meta['language']}, Year: {meta['year']}")

**Query:** modern programming features


### Results (No Filtering):


**1.** Large language models can now generate and refactor code with new programming paradigms.

   - Category: AI, Language: english, Year: 2024


**2.** Neural networks enable intelligent auto-complete and modern coding features in IDEs.

   - Category: AI, Language: english, Year: 2023


**3.** React 18 adds concurrent rendering features.

   - Category: programming, Language: javascript, Year: 2022

In [6]:
# Search with category filter: only "programming"
results_filtered = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"category": "programming"}
)

md(f"**Query:** {query}")
md("\n### Results (Filtered by category='programming'):")
for i, (doc, meta) in enumerate(zip(results_filtered['documents'][0], results_filtered['metadatas'][0]), 1):
    md(f"\n**{i}.** {doc}")
    md(f"   - Category: {meta['category']}, Language: {meta['language']}, Year: {meta['year']}")

**Query:** modern programming features


### Results (Filtered by category='programming'):


**1.** React 18 adds concurrent rendering features.

   - Category: programming, Language: javascript, Year: 2022


**2.** Python 3.11 introduces significant performance improvements.

   - Category: programming, Language: python, Year: 2023


**3.** TypeScript adds static typing to JavaScript.

   - Category: programming, Language: typescript, Year: 2012

## 4. Filter by Multiple Criteria

In [9]:
# Filter by category AND year
results_multi = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={
        "$and": [
            {"category": "AI"},
            {"year": {"$gte": 2018}}  # Greater than or equal to 2018
        ]
    }
)

md(f"**Query:** {query}")
md("\n### Results (Filtered by category='AI' AND year >= 2018):")
for i, (doc, meta) in enumerate(zip(results_multi['documents'][0], results_multi['metadatas'][0]), 1):
    md(f"\n**{i}.** {doc}")
    md(f"   - Category: {meta['category']}, Language: {meta['language']}, Year: {meta['year']}")

**Query:** modern programming features


### Results (Filtered by category='AI' AND year >= 2018):


**1.** Large language models can now generate and refactor code with new programming paradigms.

   - Category: AI, Language: english, Year: 2024


**2.** Neural networks enable intelligent auto-complete and modern coding features in IDEs.

   - Category: AI, Language: english, Year: 2023


**3.** AI-powered code completion tools are transforming modern software development.

   - Category: AI, Language: english, Year: 2023

## 5. Get All Documents by Metadata

In [11]:
# Get all AI documents
ai_docs = collection.get(
    where={"category": "AI"}
)

md("### All AI Documents:")
for doc, meta in zip(ai_docs['documents'], ai_docs['metadatas']):
    md(f"\n- {doc}")
    md(f"  - Year: {meta['year']}")

### All AI Documents:


- AI-powered code completion tools are transforming modern software development.

  - Year: 2023


- Large language models can now generate and refactor code with new programming paradigms.

  - Year: 2024


- Neural networks enable intelligent auto-complete and modern coding features in IDEs.

  - Year: 2023

## 6. Common Metadata Patterns

In [12]:
md("### Common Metadata Use Cases:")
md("""
| Metadata Field | Use Case | Example |
|----------------|----------|----------|
| **category** | Filter by topic | 'AI', 'programming', 'science' |
| **date/year** | Filter by recency | 2023, '2024-01-15' |
| **author** | Filter by source | 'OpenAI', 'Google' |
| **language** | Filter by language | 'en', 'es', 'python' |
| **source** | Track document origin | 'manual', 'wikipedia', 'docs' |
| **document_type** | Filter by format | 'pdf', 'webpage', 'code' |
""")

### Common Metadata Use Cases:


| Metadata Field | Use Case | Example |
|----------------|----------|----------|
| **category** | Filter by topic | 'AI', 'programming', 'science' |
| **date/year** | Filter by recency | 2023, '2024-01-15' |
| **author** | Filter by source | 'OpenAI', 'Google' |
| **language** | Filter by language | 'en', 'es', 'python' |
| **source** | Track document origin | 'manual', 'wikipedia', 'docs' |
| **document_type** | Filter by format | 'pdf', 'webpage', 'code' |


## 🎯 Summary

### Key Takeaways

1. **Metadata** — additional structured information (category, date, author) stored alongside documents
2. **Metadata extraction** — identifies and stores relevant attributes to enrich document records
3. **Metadata filtering** — narrows search results based on specific criteria for more precise retrieval
4. **Combined search** — pairing semantic search with metadata filtering yields the most relevant results